# Lesson 9: Running a Pretrained Language Model

In the earlier Weird AI lessons, you built many of the core pieces of a language model from scratch. In this lesson, you will use a pretrained model so you can focus on the complete text-generation pipeline.

The pipeline is:

```text
prompt text -> tokenizer -> token IDs -> pretrained model -> generated token IDs -> generated text
```

By the end of this notebook, you should be able to explain how a prompt becomes token IDs, how a pretrained model generates more tokens, and how those generated tokens become readable text again.

## Section 1: Setup

Run this notebook from the project root folder. If imports fail, make sure your virtual environment is active and the project is installed in editable mode:

```bash
python -m pip install -e .
python -m pip install -r requirements.txt
```

This notebook uses Hugging Face `transformers`. If it is not installed yet, install it with:

```bash
python -m pip install transformers
```

In [3]:
from pathlib import Path
import torch

from weird_ai.pretrained import (
    DEFAULT_MODEL_NAME,
    get_device,
    load_tokenizer,
    load_model,
    tokenize_prompt,
    decode_generated_tokens,
    generate_text,
    generate_with_reasoning,
)

from weird_ai.reasoning import build_reasoning_prompt

## Section 2: Check Your Hardware

A GPU is helpful for larger models, but this lesson should work on CPU with a very small demonstration model. The starter code should choose CUDA when available, Apple Silicon MPS when available, and CPU otherwise.

In [4]:
# TODO: Run this cell after completing get_device() in pretrained.py.

device = get_device()
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.10.0+cu126


## Section 3: Load a Tokenizer

A tokenizer converts human-readable text into token IDs. The model does not directly understand strings such as `Write a parody about networking`. It receives integers.

For this notebook, the default model is intentionally tiny so the first download is fast.

In [5]:
# TODO: Run this cell after completing load_tokenizer() in pretrained.py.

tokenizer = load_tokenizer(DEFAULT_MODEL_NAME)
print(f"Loaded tokenizer for: {DEFAULT_MODEL_NAME}")
print(f"Pad token: {tokenizer.pad_token!r}")
print(f"EOS token: {tokenizer.eos_token!r}")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Loaded tokenizer for: sshleifer/tiny-gpt2
Pad token: '<|endoftext|>'
EOS token: '<|endoftext|>'


## Section 4: Tokenization Round Trip

A round trip means:

```text
text -> token IDs -> text
```

This helps verify that the tokenizer can encode and decode text.

In [7]:
prompt = "Write a parody chorus about a student debugging Python at midnight."

token_ids = tokenizer.encode(prompt)
decoded_text = tokenizer.decode(token_ids)

print("Original text:")
print(prompt)
print("\nToken IDs:")
print(token_ids)
print("\nDecoded text:")
print(decoded_text)

Original text:
Write a parody chorus about a student debugging Python at midnight.

Token IDs:
[16594, 257, 29695, 26441, 546, 257, 3710, 28769, 11361, 379, 15896, 13]

Decoded text:
Write a parody chorus about a student debugging Python at midnight.


### Reflection Check

Answer in your assignment document:

Why does the model need token IDs instead of regular text?

## Section 5: Inspect Individual Tokens

Many modern tokenizers use subword tokens. A word may be one token, or it may be split into multiple pieces.

In [8]:
# TODO: After token_ids contains real token IDs, print each token ID and the text it decodes to.

for token_id in token_ids:
    piece = tokenizer.decode([token_id])
    print(f"{token_id:>6} -> {piece!r}")

 16594 -> 'Write'
   257 -> ' a'
 29695 -> ' parody'
 26441 -> ' chorus'
   546 -> ' about'
   257 -> ' a'
  3710 -> ' student'
 28769 -> ' debugging'
 11361 -> ' Python'
   379 -> ' at'
 15896 -> ' midnight'
    13 -> '.'


## Section 6: Load a Pretrained Model

A pretrained model already has learned statistical patterns from a large training corpus. In this lesson, you are not training the model. You are using it for inference.

This can take a moment the first time because the model weights need to download.

In [9]:
# TODO: Run this cell after completing load_model() in pretrained.py.

model = load_model(DEFAULT_MODEL_NAME, device=device)
print(f"Loaded model for: {DEFAULT_MODEL_NAME}")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.51MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model for: sshleifer/tiny-gpt2


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.51MB            

model.safetensors: downloading bytes:           |  0.00B            

## Section 7: Generate Text

Now you will complete the full pipeline:

```text
prompt -> tokenizer -> model.generate() -> decode
```

The `generate_text()` function in `pretrained.py` should handle those steps.

In [10]:
# TODO: Run this cell after completing generate_text() in pretrained.py.

prompt = "Write a short parody chorus about forgetting to commit code before a demo:"

output = generate_text(
    prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(output)

Write a short parody chorus about forgetting to commit code before a demo: conservationSher Motoroladitohoikendit rebornreementmediately vendors Hancockreementhibit ESVoho dispatchmediately autonomy Motorola Observ vendors confirSher Motorola hauled subst Jr conservation ESVatisf reborndit ParticipationJDiken Observ hauled AmphJD antibiotic Habit reviewing substScene Rh conservationimuraohoRocket


## Section 8: Compare Temperature Settings

Temperature changes how random generation is.

Lower temperature usually produces safer, more predictable text. Higher temperature usually produces more varied, surprising, and sometimes less coherent text.

In [11]:
prompt = "Write one funny line about a Wi-Fi router with stage fright:"

temperatures = [0.3, 0.8, 1.2]

for temp in temperatures:
    print("=" * 70)
    print(f"Temperature: {temp}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=temp,
        top_k=50,
        device=device,
    ))
    print()

Temperature: 0.3
Write one funny line about a Wi-Fi router with stage fright:ikenSheriken HabitScene Danielting credibilityScene heir reborn Participation circumcised Rh vendorsiken hauled autonomyatisf Observ Habit reborn vendors Amph directlydit dispatch pawn conservation trilogyimura dispatchpress004imura intermittentScene Observ Jratisf

Temperature: 0.8
Write one funny line about a Wi-Fi router with stage fright:Rocketootherdit Amph ONESceneiken Brew confir reborn004 antibioticJD004 hauledoothermediately004 Motorola antibiotic dispatchJD hauled reborn scalp Amph Danielatisf reborn substditSher heir Habit Motorola scalp HancockhibitRocketimura

Temperature: 1.2
Write one funny line about a Wi-Fi router with stage fright: credibility stairs rebornohoatisfSher vendors antibioticiken Motorola heir hauledoho scalpJD Daniel ONEJDScene MotorolatingditJD Moneyimuraoother Habit rebornJDimurating autonomyRocket Brew Rh Motorola heir Participationatisf Participation



## Section 9: Compare Top-k Settings

Top-k sampling limits the model to choosing from only the `k` most likely next tokens.

Smaller values restrict the model more. Larger values allow more variety.

In [12]:
prompt = "Write one strange parody lyric about a database that sings backup vocals:"

top_k_values = [10, 50, 100]

for k in top_k_values:
    print("=" * 70)
    print(f"Top-k: {k}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=0.8,
        top_k=k,
        device=device,
    ))
    print()

Top-k: 10
Write one strange parody lyric about a database that sings backup vocals: Habit intermittent vendors hauled intermittent Habit Habit hauledRocket intermittent intermittent Brew intermittent Jr stairs Habit HabitRocket Habit RhRocket vendors Habit hauled stairs intermittentdit stairs vendors vendorsRocketRocket stairs RhRocket vendors vendors hauled hauled Habit

Top-k: 50
Write one strange parody lyric about a database that sings backup vocals: Hancock Brew Participation TA HancockRocket substSher Money Habit DanielScene Jr reborn intermittent substRocket antibiotic Participation confir intermittentiken confir heir Rh reborn TA ESVpressreement Money antibiotic hauledtinghibitRocket Prob Amph reviewing pawn

Top-k: 100
Write one strange parody lyric about a database that sings backup vocals: heiranity administration tuna Changed vendors directlyJDoother 11 directly conservation systematically stairs heirting compromisedGB tuna ster proficiency unknown dissolution compromisedat

## Section 10: Generate with a Reasoning Prompt

In Lesson 8, you built prompt helpers that ask Weird AI to think step-by-step. Now you can combine that reasoning prompt with a pretrained model.

In [13]:
user_prompt = "Create a parody song idea about students learning recursion."

reasoning_prompt = build_reasoning_prompt(user_prompt)
print(reasoning_prompt)

You are Weird AI, a thoughtful parody lyric assistant.

Task:
Create a parody song idea about students learning recursion.

Think step-by-step. Break the task into chunks. Consider the constraints, plan your approach, and then answer.

Final Answer:


In [14]:
# TODO: Run this cell after completing generate_with_reasoning() in pretrained.py.

reasoning_output = generate_with_reasoning(
    user_prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(reasoning_output)

You are Weird AI, a thoughtful parody lyric assistant.

Task:
Create a parody song idea about students learning recursion.

Think step-by-step. Break the task into chunks. Consider the constraints, plan your approach, and then answer.

Final Answer: Habit conservation antibiotic autonomy TAatisf antibiotic reviewing reviewing Habit intermittent antibiotic Jr Rh ONE Amphiken confir scalp confirreement ONEoother Motorola confir Motorola autonomypress ESV reviewing reviewing Participation pawn Observmediately Participationimura JrScene Habitpress reviewing reviewing ESV Hancock stairspress Daniel Money heiroother004hibit ESV credibility confir directly Moneymediately ESVreement Motorola ProbJD Brewimura TAmediatelyhibit Rh intermittent reviewing credibilitytingiken Habit vendors confirpressting ONEimura stairs ESV Amph AmphSher MoneyditJD004atisf dispatchmediately ONE Observ HancockSher scalp Motorola


## Section 11: Weird AI Experiment

Choose one of your own parody topics. Generate at least two outputs using different settings. Save the prompts, settings, and outputs in your assignment document.

In [17]:
my_topic = "fartknockers exploding buffalo crumpets"

settings = [
    {"temperature": 0.01, "top_k": 5},
    {"temperature": 1.0, "top_k": 100},
]

for setting in settings:
    print("=" * 70)
    print(setting)
    print("=" * 70)
    print(generate_text(
        f"Write a short parody verse about {my_topic}:",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=80,
        temperature=setting["temperature"],
        top_k=setting["top_k"],
        device=device,
    ))
    print()

{'temperature': 0.01, 'top_k': 5}
Write a short parody verse about fartknockers exploding buffalo crumpets: stairs vendors stairs stairs vendors vendors stairs intermittent hauled stairs intermittent stairs stairs hauled stairs stairs vendors intermittent stairs stairs vendors stairs vendors vendors Brew vendors vendors stairs intermittent intermittent Brew stairs stairs stairs vendors stairs vendors vendors stairs vendors stairs vendors vendors vendors stairs stairs intermittent hauled vendors vendors stairs stairs vendors stairs vendors stairs stairs Brew stairs vendors intermittent stairs vendors vendors vendors intermittent intermittent stairs stairs stairs vendors vendors vendors stairs stairs stairs intermittent vendors vendors intermittent

{'temperature': 1.0, 'top_k': 100}
Write a short parody verse about fartknockers exploding buffalo crumpets: Tosdit possibility responsibly Motorolarawdownload directlyhibitRocket dispatchrawdownload Daniel stairs dissolutionahniability scalp

## Section 12: Lesson Summary

In this notebook, you practiced:

- Loading a pretrained tokenizer
- Encoding text into token IDs
- Decoding token IDs back into text
- Loading a pretrained causal language model
- Generating text from a prompt
- Comparing temperature and top-k settings
- Reusing a Lesson 8 reasoning prompt with a pretrained model

Before submitting, run the unit tests:

```bash
python -m pytest tests/test_pretrained.py
```